# 25.43 Laboratorio de Procesamiento de Señales — Denoising, sparsity y compresión

Clase 9. **No se entrega.**

Cada experimento arranca con una pregunta. **Escribí tu respuesta antes de correr la celda.**

El ruido de hoy lo agregás vos: así el error se puede medir contra la verdad, que es el lujo
que un registro real nunca te da.

---

## Cómo se usa

Corré la celda de abajo primero. Después, cada experimento en orden.

Si algo no corre, Discord.

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy import signal
from IPython.display import Audio, display

plt.rcParams["figure.figsize"] = (11, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

rng = np.random.default_rng(2543)

# la señal de la figura de la clase: una senoidal lenta con un pulso encima
N = 2048
n = np.arange(N)
SUBE, BAJA = int(0.55 * N) // 2 * 2 + 1, int(0.80 * N) // 2 * 2 + 1
limpia = np.sin(2 * np.pi * n / N + 0.3) + 1.6 * ((n >= SUBE) & (n < BAJA))

print("Listo.")

---

## 1. Denoising con SNR controlado

La señal de la figura de la clase —una senoidal lenta con un pulso encima—, más ruido blanco de
$\sigma = 0{,}3$ que conocemos porque lo pusimos nosotros. La cadena de la clase: transformar,
umbralar blando con el umbral universal $\lambda = \hat{\sigma}\sqrt{2\log N}$,
antitransformar.

> **Predecí:** el método tiene que estimar $\sigma$ **mirando solo la señal ruidosa**. ¿De qué
> parte de la transformada lo sacarías?

In [ ]:
SIGMA = 0.3
ruidosa = limpia + SIGMA * rng.standard_normal(N)

coefs = pywt.wavedec(ruidosa, "db4", level=5)

# sigma, estimado del nivel más fino: mediana de |d1| / 0.6745
sigma_est = np.median(np.abs(coefs[-1])) / 0.6745
lam = sigma_est * np.sqrt(2 * np.log(N))

# umbral blando sobre los detalles; la aproximación no se toca
coefs_u = [coefs[0]] + [pywt.threshold(d, lam, mode="soft") for d in coefs[1:]]
estimada = pywt.waverec(coefs_u, "db4")[:N]

print(f"sigma verdadero: {SIGMA}   estimado por MAD: {sigma_est:.3f}")
print(f"umbral universal: {lam:.3f}")
print(f"MSE de la ruidosa:  {np.mean((ruidosa - limpia)**2):.4f}")
print(f"MSE de lo estimado: {np.mean((estimada - limpia)**2):.4f}")

fig, ax = plt.subplots()
ax.plot(n, ruidosa, lw=0.7, alpha=0.5, label="ruidosa")
ax.plot(n, estimada, lw=1.6, label="umbralado blando")
ax.plot(n, limpia, lw=1.0, ls="--", color="k", alpha=0.6, label="la verdad")
ax.legend()
ax.set_xlabel("n")
plt.tight_layout()
plt.show()

El $\hat{\sigma}$ salió del **nivel más fino**: en $d_1$ la señal casi no vive —lo suave
no llega y los saltos son dos coeficientes— así que lo que hay ahí es ruido casi puro. La
mediana en vez del desvío es para que esos dos coeficientes grandes no la muevan: es la
estimación **robusta** de la clase.

Y el error cayó a una fracción del de la señal ruidosa, sin que le dijéramos nada del ruido:
ni $\sigma$, ni dónde están los saltos. El umbral universal lo dedujo todo de la propia
transformada.

---

## 2. El umbral, contra un pasabajos

El rival natural: un pasabajos, que también «saca el ruido». Para que la pelea sea justa, lo
dejamos elegir su mejor versión: barremos la frecuencia de corte y nos quedamos con la de
**mínimo MSE contra la verdad** — un lujo que solo tenemos porque el ruido lo pusimos nosotros.

Y entran los dos umbrales de la clase: el blando de recién, y el **duro**.

> **Predecí:** el mejor pasabajos posible, ¿le gana a los umbrales? ¿Dónde va a fallar cada uno?

In [ ]:
cortes = np.linspace(0.005, 0.2, 80)
mses = []
for fc in cortes:
    sos = signal.butter(4, fc, output="sos")
    mses.append(np.mean((signal.sosfiltfilt(sos, ruidosa) - limpia) ** 2))
mses = np.array(mses)

fc_opt = cortes[mses.argmin()]
pasabajos = signal.sosfiltfilt(signal.butter(4, fc_opt, output="sos"), ruidosa)

dura = pywt.waverec([coefs[0]] + [pywt.threshold(d, lam, mode="hard") for d in coefs[1:]],
                    "db4")[:N]

print(f"mejor corte: {fc_opt:.3f} (en unidades de Nyquist)")
for nombre, v in [("mejor pasabajos", pasabajos), ("umbral blando", estimada),
                  ("umbral duro", dura)]:
    print(f"MSE {nombre:16s} {np.mean((v - limpia)**2):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(cortes, mses, lw=1.4, label="pasabajos, según su corte")
axes[0].axhline(np.mean((estimada - limpia) ** 2), color="C1", lw=1.4, label="umbral blando")
axes[0].axhline(np.mean((dura - limpia) ** 2), color="C2", lw=1.4, label="umbral duro")
axes[0].set_xlabel("frecuencia de corte")
axes[0].set_ylabel("MSE")
axes[0].legend()

zona = slice(SUBE - 70, SUBE + 70)
axes[1].plot(n[zona], pasabajos[zona], lw=1.4, label="el mejor pasabajos")
axes[1].plot(n[zona], dura[zona], lw=1.2, alpha=0.8, label="umbral duro")
axes[1].plot(n[zona], estimada[zona], lw=1.4, label="umbral blando")
axes[1].plot(n[zona], limpia[zona], lw=1.0, ls="--", color="k", alpha=0.6, label="la verdad")
axes[1].set_xlabel("n — el salto, de cerca")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Mirá los números antes que el gráfico. El **duro** le ganó al mejor pasabajos, y el **blando**
quedó a un pelo — contra un rival que hizo **trampa**: su corte lo elegimos mirando la verdad,
que afuera de este notebook no existe. Los umbrales no usaron nada que la señal ruidosa no
tuviera.

El zoom muestra en qué se diferencian de verdad. El pasabajos, para alisar el ruido, **lima el
salto**: su error es una rampa sistemática que se lee como señal. Los umbrales dejan el salto
en pie —vive en pocos coeficientes grandes, que sobreviven— y su error queda repartido como
ruido residual. Y entre ellos, el compromiso de la clase: el duro conserva la amplitud pero le
pasa algún pico de ruido que superó $\lambda$; el blando es más prolijo a la vista porque
encoge todo, y ese encogimiento es sesgo.

Tres errores de tamaño parecido, tres **caracteres** de error distintos. El MSE no los
distingue — y eso importa para lo que sigue.

---

## 3. Comprimir al 50, 10, 5 y 1 %

Ahora sin ruido, y con audio: tres segundos sintéticos con las dos cosas que un audio real
mezcla — una parte **tonal** que dura, y golpes **percusivos** que no. Comprimir es el mismo
gesto que recién: quedarse con los coeficientes más grandes y tirar el resto.

> **Predecí:** ¿cuánto se puede tirar antes de que la SNR caiga de 20 dB? ¿Y a qué familia le
> va a costar más, y con qué parte del audio?

In [ ]:
fs = 8000
T = 3.0
t = np.arange(int(fs * T)) / fs

# la parte tonal: tres notas que se encadenan, con ataque y caída
audio = np.zeros_like(t)
for f0, t0 in [(330, 0.0), (415, 1.0), (494, 2.0)]:
    env = np.exp(-3 * (t - t0)) * ((t >= t0) & (t < t0 + 1.0))
    audio += env * (np.sin(2 * np.pi * f0 * t) + 0.3 * np.sin(2 * np.pi * 2 * f0 * t))

# la parte percusiva: un golpe seco cada medio segundo
for t0 in np.arange(0.25, T, 0.5):
    dur = int(0.008 * fs)
    i0 = int(t0 * fs)
    audio[i0 : i0 + dur] += 2.0 * rng.standard_normal(dur) * np.exp(-np.arange(dur) / (0.002 * fs))

audio /= np.abs(audio).max()


def comprimir(x, familia, fraccion):
    coefs = pywt.wavedec(x, familia, level=6)
    todos = np.concatenate(coefs)
    umbral = np.quantile(np.abs(todos), 1 - fraccion)
    coefs_c = [pywt.threshold(c, umbral, mode="hard") for c in coefs]
    return pywt.waverec(coefs_c, familia)[: len(x)]


def snr_db(x, xr):
    return 10 * np.log10(np.sum(x**2) / np.sum((x - xr) ** 2))


fracciones = np.array([0.50, 0.10, 0.05, 0.01])
fig, ax = plt.subplots()
comprimidos = {}
for familia in ["haar", "db4", "bior4.4"]:
    curva = [snr_db(audio, comprimir(audio, familia, fr)) for fr in fracciones]
    ax.plot(fracciones * 100, curva, "o-", lw=1.4, label=familia)
    comprimidos[familia] = comprimir(audio, familia, 0.05)
    print(f"{familia:8s} " + "  ".join(f"{fr:4.0%}: {s:5.1f} dB" for fr, s in zip(fracciones, curva)))

ax.set_xscale("log")
ax.set_xticks(fracciones * 100, [f"{f:g}" for f in fracciones * 100])
ax.set_xlabel("% de coeficientes retenidos")
ax.set_ylabel("SNR [dB]")
ax.set_title("La curva tasa–distorsión — la figura del TP2, sobre un audio de juguete")
ax.legend()
plt.tight_layout()
plt.show()

Al 50 % no perdiste casi nada; al 1 % la SNR se desploma. En el medio está la zona donde el
TP2 trabaja: **cuánto** se puede tirar, medido, para cada tipo de contenido y cada familia.

La curva es la de la clase — cada punto es una fracción retenida y lo que costó. Lo que la
curva no dice es *cómo suena* ese costo. Eso sigue.

---

## 4. La escucha, por familias

Los tres comprimidos del experimento 3, al **5 %**, uno por familia. La SNR ya la conocés;
ahora escuchá.

> **Predecí:** ¿qué se va a romper primero — las notas o los golpes? ¿Y en qué familia?

In [ ]:
print("original")
display(Audio(audio, rate=fs))
for familia, x in comprimidos.items():
    print(f"{familia}, al 5 % — SNR {snr_db(audio, x):.1f} dB")
    display(Audio(x, rate=fs))

Dos números de SNR parecidos pueden sonar muy distinto: el oído no promedia como el MSE — un
golpe emborronado se nota más que un error repartido.

Para probar con **tu** audio: cargalo con `scipy.io.wavfile.read`, convertilo a `float`,
normalizalo, y pasale las mismas funciones.